# exp_013 — new-methods sweep (Colab)

Sweeps the **new** intrinsic-exploration proposals (ICM/RND baselines are NOT re-run —
compare against their known numbers + the random baseline). Metric: **env-steps to first
extrinsic reward** (stop-on-first-reward; right-censored at a per-cell cap). Intrinsic-only.

**Methods** (all share γ=0.95, c_value/value-lr bump, reward-clip, non-episodic, held-out freeze):
| tag | what | module |
|---|---|---|
| **A** `rnd_leak_frozen` | RND+leak on a **frozen-random** φ (no ICM) | `exp_013_1b_leaky_rnd_on_icm_phi --phi-mode frozen` |
| **B** `rnd_leak_icm` | RND+leak on **ICM** φ | `exp_013_1b_leaky_rnd_on_icm_phi` |
| **B-xfer** | B on L2 with φ **initialised from a trained L1 φ** (cross-level transfer) | `…--init-phi-ckpt <L1 ckpt>` |
| **C** `additive` | `w·norm(ICM)+(1−w)·norm(RND)`, w=0.5 (probe says ~dead; control) | `exp_013_2_additive_rnd_icm` |
| **D** `lookahead` | MCTS-organized **1-step lookahead softmax** (actor-free; no entropy collapse) | `exp_013_3_mcts_lookahead` |
| **E** `disagreement` | Plan2Explore ensemble (optional bonus) | `exp_013_4_plan2explore` |

> **Set Runtime ▸ GPU.** See `probes/compute_optimization.md` (once generated) for the recommended n_envs / concurrency.

## 1. Setup

In [ ]:
import os, sys, glob, json, time, subprocess
REPO_URL="https://github.com/LavetteSinsora/ProjectArceus.git"; REPO="/content/ProjectArceus"
if not os.path.isdir(REPO):
    !git clone --depth 1 $REPO_URL $REPO
%cd /content/ProjectArceus
!git pull --ff-only -q || true
!pip -q install "arc-agi>=0.9.8" "arcengine>=0.9.3"
!pip -q install -e . --no-deps
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(set Runtime→GPU!)")

## 2. Sweep config (compute-efficient — edit for your budget)

In [ ]:
SEEDS = [0, 1, 2]
CONCURRENCY = 1                  # 1 = sequential; raise per probes/compute_optimization.md if the GPU has headroom
N_ENVS = None                    # None = method default (16); set higher for faster wall-clock if GPU not saturated
RUN_D = True                     # include proposal D (lookahead / MCTS)
RUN_DISAGREE = False             # also run exp_013_4 disagreement (optional; not in the original 4)

CAPS = {("ls20",0):200_000, ("ls20",1):300_000, ("tu93",0):600_000, ("re86",0):1_000_000}
RANDOM_E = {("ls20",0):"~50k", ("ls20",1):"inf", ("tu93",0):"~500k", ("re86",0):"~2.0M"}

TRANSFER_SRC = ("ls20", 0)       # train B here (also a data point) ...
TRANSFER_DST = ("ls20", 1)       # ... transfer its φ to here (vs B-random)
COMPARE_CELLS = [("ls20", 0)]    # cells for the A/C/D method comparison

EXP1 = "JEPA.experiments.exp_013_headline_experiment.exp_013_1b_leaky_rnd_on_icm_phi.run"
EXP2 = "JEPA.experiments.exp_013_headline_experiment.exp_013_2_additive_rnd_icm.run"
EXP4 = "JEPA.experiments.exp_013_headline_experiment.exp_013_4_plan2explore.run"
EXP5 = "JEPA.experiments.exp_013_headline_experiment.exp_013_3_mcts_lookahead.run"
RUNS1 = "JEPA/experiments/exp_013_headline_experiment/exp_013_1b_leaky_rnd_on_icm_phi/runs"
LOGDIR = "/content/exp013_logs"; os.makedirs(LOGDIR, exist_ok=True)
print("transfer:", TRANSFER_SRC, "->", TRANSFER_DST, "| compare on", COMPARE_CELLS, "| seeds", SEEDS, "| D=", RUN_D)

## 3. Run the sweep

Phase 1: train **B** on the transfer source (= ls20-L1 B data point + φ source), sequentially
(so the φ checkpoints exist). Phase 2: A / C / D on the compare cells, and B-random + B-xfer
(+ D) on the transfer-dst cell. Re-runnable.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def base_args(g, l):
    a = ["--game", g, "--level", str(l), "--max-env-steps", str(CAPS[(g, l)])]
    if N_ENVS: a += ["--n-envs", str(N_ENVS)]
    return a

def run_proc(name, module, extra):
    cmd = [sys.executable, "-m", module] + extra
    t0 = time.time()
    with open(f"{LOGDIR}/{name}.log", "w") as f:
        rc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT).returncode
    print(f"[{'ok ' if rc==0 else 'FAIL'}] {name:44s} {(time.time()-t0)/60:5.1f} min")
    return rc

def latest_ckpt(exp_name):
    cks = sorted(glob.glob(f"{RUNS1}/{exp_name}_*/checkpoints/step_*.pt"))
    return cks[-1] if cks else None

t_start = time.time()

# --- Phase 1: transfer SOURCE = B (icm) on ls20 L1, per seed (sequential for the φ ckpts) ---
sg, sl = TRANSFER_SRC
for s in SEEDS:
    run_proc(f"B_{sg}_L{sl+1}_s{s}", EXP1, base_args(sg, sl) + ["--seed", str(s)])
src_ck = {s: latest_ckpt(f"exp013_1_rndicm_icm_{sg}_L{sl+1}_seed{s}") for s in SEEDS}
print("source φ ckpts:", {s: (p is not None) for s, p in src_ck.items()})

# --- Phase 2 jobs ---
jobs = []
for (g, l) in COMPARE_CELLS:
    for s in SEEDS:
        sd = ["--seed", str(s)]
        jobs.append((f"A_frozen_{g}_L{l+1}_s{s}", EXP1, ["--phi-mode", "frozen"] + base_args(g, l) + sd))
        jobs.append((f"C_additive_{g}_L{l+1}_s{s}", EXP2, base_args(g, l) + sd))
        if RUN_D:       jobs.append((f"D_lookahead_{g}_L{l+1}_s{s}", EXP5, base_args(g, l) + sd))
        if RUN_DISAGREE: jobs.append((f"E_disagree_{g}_L{l+1}_s{s}", EXP4, base_args(g, l) + sd))
dg, dl = TRANSFER_DST
for s in SEEDS:
    sd = ["--seed", str(s)]
    jobs.append((f"B_random_{dg}_L{dl+1}_s{s}", EXP1, base_args(dg, dl) + sd))
    if src_ck.get(s):
        jobs.append((f"B_xfer_{dg}_L{dl+1}_s{s}", EXP1, base_args(dg, dl) + sd + ["--init-phi-ckpt", src_ck[s]]))
    if RUN_D:       jobs.append((f"D_lookahead_{dg}_L{dl+1}_s{s}", EXP5, base_args(dg, dl) + sd))

with ThreadPoolExecutor(max_workers=CONCURRENCY) as ex:
    list(ex.map(lambda j: run_proc(*j), jobs))
print(f"\nSWEEP DONE in {(time.time()-t_start)/60:.1f} min")

## 4. Aggregate vs the random benchmark

In [ ]:
import numpy as np
from collections import defaultdict

base = "JEPA/experiments/exp_013_headline_experiment"
files = []
for d in ["exp_013_1b_leaky_rnd_on_icm_phi","exp_013_2_additive_rnd_icm","exp_013_4_plan2explore","exp_013_3_mcts_lookahead"]:
    files += glob.glob(f"{base}/{d}/runs/*/result.json")
rows = [json.load(open(f)) for f in files]

def method_tag(r):
    m = r.get("method")
    if m == "rnd_icm_additive": return "C_additive"
    if m == "disagreement":     return "E_disagreement"
    if m == "lookahead_mcts":   return "D_lookahead"
    n = r["exp_name"]
    if "_frozen_" in n:   return "A_rnd_leak_frozen"
    if "_icm_xfer_" in n: return "B_xfer"
    return "B_rnd_leak_icm"

agg = defaultdict(list)
for r in rows:
    agg[(r["game"], r["level_index"], method_tag(r))].append(r)

print(f"{'game':>5} {'lvl':>3} {'method':>18} {'n':>3} {'solved':>7} {'median':>10} {'mean':>10} {'rand_E':>8}")
print("-"*72)
for k in sorted(agg, key=lambda x:(x[0],x[1],x[2])):
    rs = agg[k]; sv = np.array([r["env_steps_to_first_reward"] for r in rs if r["solved"]], float)
    med = f"{np.median(sv):,.0f}" if sv.size else "—"; mean = f"{sv.mean():,.0f}" if sv.size else "—"
    print(f"{k[0]:>5} {k[1]+1:>3} {k[2]:>18} {len(rs):>3} {len(sv)}/{len(rs):<5} {med:>10} {mean:>10} {RANDOM_E.get((k[0],k[1]),'?'):>8}")

print("\nKEY COMPARISONS: A vs B (does ICM-φ beat frozen-random φ?) | B_random vs B_xfer on L2 (L1→L2 transfer?)")
print("                 D vs A/B (does lookahead beat reward-based?) | C ~ control (likely dead)")

import csv
out=f"{base}/sweep_exp013_results.csv"
with open(out,"w",newline="") as f:
    w=csv.writer(f); w.writerow(["method","game","level_index","seed","env_steps_to_first_reward","solved","total_env_steps"])
    for r in rows: w.writerow([method_tag(r),r["game"],r["level_index"],r["seed"],r["env_steps_to_first_reward"],r["solved"],r["total_env_steps"]])
print("wrote", out, f"({len(rows)} runs)")

## 5. Download results

In [ ]:
import shutil
shutil.make_archive("/content/exp013_sweep_results", "zip", "JEPA/experiments/exp_013_headline_experiment")
try:
    from google.colab import files; files.download("/content/exp013_sweep_results.zip")
except Exception as e:
    print("Download from the Files pane: /content/exp013_sweep_results.zip", e)